In [ ]:
pip install transformers


In [ ]:
pip install bitsandbytes


In [ ]:
pip install pillow


In [ ]:
pip install "accelerate>=0.26.0"


## Loading the Quantized Model

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

# Load the processor and model
processor = AutoProcessor.from_pretrained("unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit")
model = AutoModelForImageTextToText.from_pretrained("unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit")

# Move model to GPU (if available) or keep it on CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)



## Testing the Quantized Model on Different Images

In [ ]:
# Open the image
image = Image.open("/teamspace/studios/this_studio/Screenshot 2024-10-08 211353.png")  # Replace with your image file path

# Process the image
image_input = processor(images=image, return_tensors="pt")

# Process the text input
text_input = processor(text="Extract only the text from this image without any additional interpretation or processing.", return_tensors="pt")

# Merge image and text inputs into a batch
inputs = {**image_input, **text_input}

# Move input tensors to the same device as the model
inputs = {key: value.to(device) for key, value in inputs.items()}

outputs = model.generate(
    **inputs,
    max_length=256,
    top_p=0.85,
    num_beams=1,  # Enforce deterministic output
    repetition_penalty=2.0  # Penalize repetitive sequences
)


extracted_text = processor.decode(outputs[0], skip_special_tokens=True)
cleaned_text = extracted_text.split("<OCR/>")[0]  # Discard unnecessary additions.


print("Extracted Text:", cleaned_text)
